In [1]:
from pprint import pprint

import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.express as px

from lcm import (
    AgeGrid,
    DiscreteGrid,
    LinSpacedGrid,
    LogSpacedGrid,
    Model,
    Regime,
    categorical,
)
from lcm.typing import (
    BoolND,
    ContinuousAction,
    ContinuousState,
    DiscreteAction,
    FloatND,
    ScalarInt,
)

# Generating a way easier model to solve problem with grids

In [2]:
def earnings (
    wage: float
) -> FloatND:
    return wage
    
def liquidation_cost(
    age: float,
    #zilliq: float,
) -> FloatND:
    return 0.5 / (1 + jnp.exp((age - 50) / 10))

def total_consumption(
    earnings: FloatND, 
    investment_x: ContinuousAction, 
    investment_z: ContinuousAction, 
    liquidation_cost: FloatND,
) -> FloatND:
    liq_cost = liquidation_cost * jnp.minimum(investment_z, 0)
    return earnings - investment_x - investment_z + liq_cost

def utility(
    total_consumption: FloatND,    
    risk_aversion: float,
    wealth_illiquid: ContinuousState,
) -> FloatND:
  
    """CRRA utility.
    """
    total = total_consumption + (0.05 * wealth_illiquid)
    return total ** (1 - risk_aversion) / (1 - risk_aversion)

def beq_utility(
        mean_hhs: float,
        mean_hhy: float,
        risk_aversion: float,
        wealth: ContinuousState,
        wealth_illiquid: ContinuousState,
        liquidation_cost: FloatND,
        interest_rate: float,
        alpha: float,
        discount_factor: float,
) -> FloatND:
        """
        Utility when agent dies.
        """
        # liquidation_cost = 1/3 if zilliq=1
        beq = wealth + (wealth_illiquid * (1-liquidation_cost))
        beq_annuity = interest_rate * (jnp.maximum(beq, 0)) # usan en matlab (jnp.maximum(interest_rate-1, 0)) porque puede ser tasa negativa... pero mi tasa 1 no es negativa, 2, seria mas bien maximo entre la tasa y 0 porque no es bruta la tasa que pongo
        u_baseline = mean_hhs * (((mean_hhy/mean_hhs)**(1-risk_aversion))-1) / (1-risk_aversion) # me falta agregar si rho=1
        u_bequest = mean_hhs * ((((mean_hhy + beq_annuity)/mean_hhs)**(1-risk_aversion))-1) / (1-risk_aversion)
        return (alpha/(1-discount_factor)) * (u_bequest - u_baseline)



In [3]:
def end_of_period_wealth(
    wealth: ContinuousState,
    investment_x: ContinuousAction,
) -> FloatND:
    return wealth + investment_x

def next_wealth(
    end_of_period_wealth: FloatND,
    interest_rate: float,
    interest_rate_debt: float,
) -> ContinuousState:
   
    return (
        jnp.maximum(end_of_period_wealth, 0) * (1 + interest_rate)
        + jnp.minimum(end_of_period_wealth, 0) * (1 + interest_rate_debt) 
    )


### Illiquid Wealth Z ###

def end_of_period_wealth_illiquid(
    wealth_illiquid: ContinuousState,
    investment_z: ContinuousAction,
) -> FloatND:
    """Illiquid wealth, before interest."""
    return wealth_illiquid + investment_z
    

def next_wealth_illiquid(
    end_of_period_wealth_illiquid: FloatND,
    interest_rate_illiquid: float,
) -> ContinuousState:
    return (end_of_period_wealth_illiquid) * (1 + interest_rate_illiquid)



In [4]:
"""def budget_constraint(
        earnings: FloatND,
        investment_z: ContinuousAction,
        investment_x: ContinuousAction,
        liquidation_cost: FloatND,   
    ) -> BoolND:
   
    liq_cost = liquidation_cost * jnp.minimum(investment_z, 0)
    consumption = earnings - investment_x - investment_z + liq_cost
    return consumption >= 0 
"""
def budget_constraint(
        total_consumption: FloatND,
    ) -> BoolND:
    """
    Consumption needs to be positive
    """
    return total_consumption > 0 

def borrowing_constraint(
    end_of_period_wealth: FloatND,
) -> BoolND:
    """
    Simple borrowing constraint.
    """
    return end_of_period_wealth >= - 51

def illiquid_wealth_constraint(
    end_of_period_wealth_illiquid: FloatND,
) -> BoolND:
    """
    Total illiquid cannot be negative.
    """
    return end_of_period_wealth_illiquid >= 0 

def ponzi_constraint(
    end_of_period_wealth: FloatND,
    age: float,
) -> BoolND:
    """
    Liquid wealth before dying cannot be negative (if we do not add this, debt explodes, see figure 1).
    """

    return jnp.where(age == 85, end_of_period_wealth>=0, True)

In [5]:
def next_regime(age: float, last_working_age: float, final_age: float) -> ScalarInt:
    return jnp.where(
        age >= final_age,
        RegimeId.dead,
        jnp.where(
            age >= last_working_age,
            RegimeId.retirement,
            RegimeId.working_life,
        )
    )

In [6]:
@categorical(ordered=False)
class RegimeId:
    working_life: int
    retirement: int
    dead: int

In [32]:
age_grid = AgeGrid(start=25, stop=105, step="20Y")
retirement_age = age_grid.exact_values[-3]
dead_age = age_grid.exact_values[-1]


wealth_liquid_grid = LinSpacedGrid(start=-50, stop=50, n_points=50)
wealth_illiquid_grid = LinSpacedGrid(start=0.1, stop=100, n_points=50)
investment_x_grid = LinSpacedGrid(start=-50, stop=50, n_points=200)
investment_z_grid = LinSpacedGrid(start=-50, stop=50, n_points=200)

working_life = Regime(
    transition=next_regime,
    active=lambda age: age < retirement_age,
    states={
        "wealth": wealth_liquid_grid,
        "wealth_illiquid": wealth_illiquid_grid,
    },
    state_transitions={
        "wealth": next_wealth,
        "wealth_illiquid": next_wealth_illiquid,
    },
    actions={
        "investment_x": investment_x_grid,
        "investment_z": investment_z_grid,
    },
    functions={
        "total_consumption": total_consumption,
        "utility": utility,
        "liquidation_cost": liquidation_cost,
        "earnings": earnings,
        "end_of_period_wealth": end_of_period_wealth,
        "end_of_period_wealth_illiquid": end_of_period_wealth_illiquid,
    },
    constraints={
        "borrowing_constraint": borrowing_constraint,
        "illiquid_wealth_constraint": illiquid_wealth_constraint,
        "budget_constraint": budget_constraint,
        #"ponzi_constraint": ponzi_constraint
    },
)

retirement = Regime(
    transition=next_regime,
    active=lambda age: (age >= retirement_age) & (age < dead_age),
    states={
        "wealth": wealth_liquid_grid,
        "wealth_illiquid": wealth_illiquid_grid,
    },
    state_transitions={
        "wealth": next_wealth,
        "wealth_illiquid": next_wealth_illiquid,
    },
     actions={
        "investment_x": investment_x_grid,
        "investment_z": investment_z_grid,
    },
    functions={
        "total_consumption": total_consumption,
        "utility": utility,
        "liquidation_cost": liquidation_cost,
        "earnings": earnings,
        "end_of_period_wealth": end_of_period_wealth,
        "end_of_period_wealth_illiquid": end_of_period_wealth_illiquid,
    },
    constraints={
        "borrowing_constraint": borrowing_constraint,
        "illiquid_wealth_constraint": illiquid_wealth_constraint,
        "budget_constraint": budget_constraint,
        "ponzi_constraint": ponzi_constraint
    },
)

dead = Regime(
    transition=None,
    active=lambda age: True,
    functions={
        "utility": beq_utility,
        "liquidation_cost": liquidation_cost,
        },
    states={
        "wealth": wealth_liquid_grid,
        "wealth_illiquid": wealth_illiquid_grid,
    },
)

model = Model(
    regimes={
        "working_life": working_life,
        "retirement": retirement,
        "dead": dead,
    },
    ages=age_grid,
    regime_id_class=RegimeId,
    description="Lifecycle consumption-savings model.",
)


In [24]:
params = {
    "discount_factor":    0.96, # is it a problem that dead does not need anything and I still add them here?
    "risk_aversion":      1.5,
    "interest_rate":      0.0203,
    "interest_rate_debt": 0.1059, 
    "interest_rate_illiquid": 0, # habia puesto 1 y esto hacia que explotara porque
    "working_life": {
        "next_regime": {
            "last_working_age": retirement_age -20, #OJO NO ME MUESTRA ESTO EN EL DICC Y ES REPETITIVO (4 hmvg)
            "final_age":        dead_age - 20,
        },
        "earnings": {
            "wage": 60.0
        }
    },
     "retirement": {
         "next_regime": {
            "last_working_age": retirement_age -20,
            "final_age":        dead_age - 20,
        },
        "earnings": {
            "wage": 60.0
        }
    },
    "dead": {
        "utility": {
            "alpha": 0.5, # debe haber una forma de no tener que volver a escribir todos estos numeros
            #"delta": 0.96,
            #"interest_rate": 0.02,
            "mean_hhs": 2,
            "mean_hhy": 1000, 
            #"risk_aversion": 2.0,
            }}
}


In [33]:
n_agents = 100
result = model.simulate(
    params=params, log_level="debug", log_path="./debug/",
    initial_conditions={
        "regime": np.zeros(n_agents, dtype=int),
        "age": np.full(n_agents, float(age_grid.exact_values[0])),  
        "wealth": np.full(n_agents, 10),
        "wealth_illiquid": np.full(n_agents, 10),   
    },
    period_to_regime_to_V_arr=None,
)

AOT compilation: 5 unique functions (9 regime-period pairs, 32 workers)
1/5  working_life (age 25)
  lowering ...


  lowered in 71.0ms
2/5  working_life (age 45)
  lowering ...
  lowered in 64.1ms
3/5  retirement (age 65)
  lowering ...
  lowered in 73.1ms
4/5  retirement (age 85)
  lowering ...
  lowered in 58.3ms
5/5  dead (age 25)
  lowering ...
  lowered in 34.0ms
  compiling working_life (age 25) ...
  compiling working_life (age 45) ...
  compiling retirement (age 85) ...
  compiling retirement (age 65) ...
  compiling dead (age 25) ...
  compiled  dead (age 25)  139.2ms
  compiled  retirement (age 85)  708.6ms
  compiled  retirement (age 65)  717.5ms
  compiled  working_life (age 25)  733.3ms
  compiled  working_life (age 45)  730.2ms
Starting solution
Age 105 (1 regimes):
  - dead: V min=0 max=0.00339 mean=0.00118
  finished in 8.8ms
Age 85 (2 regimes):
NaN/Inf in V_arr for regime 'retirement' at age 85
  - retirement: V min=nan max=nan mean=nan


InvalidValueFunctionError: Value function at age 85.0 in regime 'retirement': all values are NaN.

NaN propagates through Q = U + beta * E[V]. Common causes:
- A missing feasibility constraint (e.g. negative leisure passed to a fractional exponent).
- A regime parameter is NaN.
- The utility function returned NaN (e.g. log of a non-positive argument).
- The regime transition function returned NaN probabilities (e.g. from a NaN survival probability or a NaN fixed param).
- A per-target state_transitions dict omits a reachable target (non-zero transition probability to an incomplete target).

To diagnose, re-solve with debug logging:

  model.solve(params=params, log_level="debug", log_path="./debug/")

The snapshot saved on failure contains diagnostics that pinpoint where NaN enters (U, E[V], or regime transitions). See the debugging guide:
https://pylcm.readthedocs.io/en/latest/user_guide/debugging/

# Understanding the problem

In [16]:
import h5py
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

path_hf_file = "/home/brto/brenda/master-thesis/src/replication_laibsonetal/life_cycle_model/debug/simulate_snapshot_016/arrays.h5"
with h5py.File(path_hf_file, "r") as f:
    print(list(f.keys()))

['0', '1', '2', '3', '4']


In [11]:
with h5py.File(path_hf_file, "r") as f:
    print(list(f["2"].keys()))

['retirement']


In [17]:
with h5py.File(path_hf_file, "r") as f:
    
    for period in f.keys(): 
        for regime in f[period].keys():  
            
            V_arr = f[period][regime]["V_arr"][:]
            
            n_nan = np.isnan(V_arr).sum()
            n_inf = np.isinf(V_arr).sum()
            
            if n_nan > 0 or n_inf > 0:
                print(
                    f"Period {period}, regime '{regime}': "
                    f"shape={V_arr.shape}, NaN={n_nan}, Inf={n_inf}"
                )

In [13]:
with h5py.File(path_hf_file, "r") as f:
    V = f["3/retirement/V_arr"][:]
    # Shape: (25, 25) → wealth x wealth_illiquid

fig = go.Figure(data=go.Heatmap(
    z=V,
    colorscale="Viridis",
    colorbar=dict(title="V"),
))
fig.update_layout(
    title="Value function — retirement, age 85",
    xaxis_title="wealth index",
    yaxis_title="wealth_illiquid index",
)
fig.show()

# The following code was for later analysis when there is no error

In [30]:
df = result.to_dataframe(additional_targets="all")
df["age"] = df["age"].astype(int)
print(df)

     subject_id  period        regime     value     wealth  wealth_illiquid  \
0             0       0  working_life -0.932098  10.000000        10.000000   
1             0       1  working_life -0.712550   3.281366        12.261307   
2             0       2    retirement -0.481050   0.015339        12.512566   
3             0       3    retirement -0.243426  -5.262458        12.763824   
4             0       4          dead  0.000012   0.014207         0.452269   
..          ...     ...           ...       ...        ...              ...   
495          99       0  working_life -0.932098  10.000000        10.000000   
496          99       1  working_life -0.712550   3.281366        12.261307   
497          99       2    retirement -0.481050   0.015339        12.512566   
498          99       3    retirement -0.243426  -5.262458        12.763824   
499          99       4          dead  0.000012   0.014207         0.452269   

     investment_x  investment_z  age borrowing_cons

In [31]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

df_mean = df.groupby("age", as_index=False).mean(numeric_only=True)
df_mean["consumption"] = df_mean["earnings"] - df_mean["investment_x"] - df_mean["investment_z"] #+ df_mean["liquidation_cost"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_mean["age"], y=df_mean["earnings"], name="Income", line=dict(color='firebrick', width=3)))
fig.add_trace(go.Scatter(x=df_mean["age"], y=df_mean["consumption"], name="Total Consumption", line=dict(color='royalblue', width=3)))
fig.add_trace(go.Scatter(x=df_mean["age"], y=df_mean["wealth"], name="Liquid Assets", line=dict(color='forestgreen', width=3)))
fig.add_trace(go.Scatter(x=df_mean["age"], y=df_mean["wealth_illiquid"], name="Illiquid Assets/10", line=dict(color='goldenrod', width=3)))
# borre /10 porque no es ni tan grande al dividir...
fig.update_layout(
    title="Average Lifecycle Profile",
    xaxis_title="Age",
    #yaxis_title="Units (x 10^4)",
    template="plotly_white", # Fondo blanco como en la foto
    legend=dict(
        x=0.1, y=0.9,
        bgcolor='rgba(255, 255, 255, 0.5)',
        bordercolor="Black",
        borderwidth=1
    )
)

fig.show()

Si tenemos un bequest con mean_hhy suficientemente alto, funciona.
Pero lo mas IMPORTANTE, es que restringi el credito.